# Практика · ML Handbook гл. 44 (44 nn optimization)

Источник: [nn-optimization.html](https://mlhandbook-production.up.railway.app/parts/deep-learning/nn-optimization.html)

Теория: [../d.5.44 Оптимизация нейросетей — дистилляция квантование прунинг.md](../d.5.44 Оптимизация нейросетей — дистилляция квантование прунинг.md) · [README.md](./README.md)

**Handbook-код — в этом файле**. Эталон: [handbook_gl44_практика_решения.ipynb](./handbook_gl44_практика_решения.ipynb) **после** попытки.

| ☐ | Блок |\n|---|------|\n| ☐ | §44.1 quantize_dynamic |\n| ☐ | §44.2 prune |\n| ☐ | §44.3 distillation_loss |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


---
# §44.3 · distillation loss



In [ ]:
def distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.3):
    hard = F.cross_entropy(student_logits, labels)
    soft = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean',
    ) * (T * T)
    return alpha * hard + (1 - alpha) * soft

s_logits = torch.randn(8, 10, requires_grad=True)
t_logits = torch.randn(8, 10)
labels = torch.randint(0, 10, (8,))
print('distill loss', round(distillation_loss(s_logits, t_logits, labels).item(), 3))


---
# §44.2 · prune 60%



In [ ]:
import torch.nn.utils.prune as prune
layer = nn.Linear(100, 100)
prune.l1_unstructured(layer, name='weight', amount=0.6)
sp = (layer.weight == 0).float().mean().item()
print('sparsity', f'{sp:.0%}')


---
# §44.1 · quantize_dynamic



In [ ]:
import torch.ao.quantization as tq
model = nn.Sequential(nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 10))
q = tq.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
print('quantized model ready')
